# Módulo 03 · Equilibrio de Nash
**Teoría de Juegos — Tutorial Interactivo**

El equilibrio de Nash es el concepto central de la teoría de juegos moderna. Usaremos `nashpy` para calcular equilibrios y `matplotlib` para visualizar las curvas de mejor respuesta.

In [ ]:
import numpy as np
import nashpy as nash
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'font.family': 'sans-serif', 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 100})
print('nashpy', nash.__version__)

## 1. Crear un juego con nashpy

Un juego se define con dos matrices: `A` (pagos J1) y `B` (pagos J2).

In [ ]:
# Batalla de Sexos
# Ambos prefieren estar juntos, pero J1 prefiere Ópera y J2 prefiere Fútbol
A_bos = np.array([[2, 0], [0, 1]])  # J1
B_bos = np.array([[1, 0], [0, 2]])  # J2

bos = nash.Game(A_bos, B_bos)
print(bos)

## 2. Encontrar todos los equilibrios (support enumeration)

`support_enumeration()` devuelve todos los equilibrios de Nash (puros y mixtos).

In [ ]:
equilibria = list(bos.support_enumeration())
print(f'Equilibrios encontrados: {len(equilibria)}')
for i, (sigma1, sigma2) in enumerate(equilibria):
    tipo = 'puro' if max(sigma1) == 1 else 'mixto'
    pago = bos[sigma1, sigma2]
    print(f'  EQ {i+1} ({tipo}): J1={np.round(sigma1, 3)}, J2={np.round(sigma2, 3)} → pagos={np.round(pago, 3)}')

## 3. Visualizar curvas de mejor respuesta

Para un juego 2×2:
- **BR(J1)**: dado que J2 juega col 0 con prob `q`, ¿cuál es la mejor respuesta de J1?
- **BR(J2)**: dado que J1 juega fila 0 con prob `p`, ¿cuál es la mejor respuesta de J2?

Los equilibrios de Nash son las **intersecciones** de las dos curvas.

In [ ]:
def plot_best_responses(A, B, row_labels, col_labels, title=''):
    fig, ax = plt.subplots(figsize=(6, 5))
    
    # Umbral q* donde J1 es indiferente entre sus dos estrategias
    # A[0,0]*q + A[0,1]*(1-q) = A[1,0]*q + A[1,1]*(1-q)
    dA = (A[0,0] - A[0,1]) - (A[1,0] - A[1,1])
    q_star = None if abs(dA) < 1e-10 else (A[1,1] - A[0,1]) / dA
    q_star = None if q_star is None else np.clip(q_star, 0, 1)
    
    # Umbral p* donde J2 es indiferente entre sus dos estrategias
    dB = (B[0,0] - B[1,0]) - (B[0,1] - B[1,1])
    p_star = None if abs(dB) < 1e-10 else (B[1,1] - B[1,0]) / dB
    p_star = None if p_star is None else np.clip(p_star, 0, 1)
    
    COLOR1, COLOR2 = '#1a3a5c', '#b85c00'
    
    # BR de J1 (en espacio p-q: la curva está en términos de q → p óptimo)
    if q_star is not None:
        ax.plot([0, 0], [0, q_star], color=COLOR1, lw=2.5, label=f'BR({row_labels[0]}, J1)')
        ax.plot([0, 1], [q_star, q_star], color=COLOR1, lw=2.5, ls='--')
        ax.plot([1, 1], [q_star, 1], color=COLOR1, lw=2.5)
    
    # BR de J2 (en espacio p-q: la curva está en términos de p → q óptimo)
    if p_star is not None:
        ax.plot([0, p_star], [0, 0], color=COLOR2, lw=2.5, label=f'BR({col_labels[0]}, J2)')
        ax.plot([p_star, p_star], [0, 1], color=COLOR2, lw=2.5, ls='--')
        ax.plot([p_star, 1], [1, 1], color=COLOR2, lw=2.5)
    
    # Equilibrios de Nash
    game = nash.Game(A, B)
    for sigma1, sigma2 in game.support_enumeration():
        p, q = sigma1[0], sigma2[0]
        color = '#2d7a50' if max(sigma1) == 1 else '#8b5cf6'
        ax.scatter([p], [q], s=120, color=color, zorder=10, edgecolors='white', linewidths=1.5)
        ax.annotate(f'NE ({p:.2f}, {q:.2f})', (p, q), (p+0.03, q+0.03), fontsize=9, color=color)
    
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel(f'p = Prob({row_labels[0]}) — J1', fontsize=11)
    ax.set_ylabel(f'q = Prob({col_labels[0]}) — J2', fontsize=11)
    ax.set_title(title or 'Curvas de mejor respuesta', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, ls=':')
    plt.tight_layout()
    return fig

_ = plot_best_responses(A_bos, B_bos, ['Ópera','Fútbol'], ['Ópera','Fútbol'], 'Batalla de Sexos')
plt.show()

## 4. Los tres juegos clásicos de coordinación

In [ ]:
juegos = {
    'Caza del Ciervo (Stag Hunt)': {
        'A': np.array([[4, 0], [2, 2]]),
        'B': np.array([[4, 2], [0, 2]]),
        'rl': ['Ciervo', 'Liebre'], 'cl': ['Ciervo', 'Liebre']
    },
    'Gallina (Chicken)': {
        'A': np.array([[-5, 3], [0, 1]]),
        'B': np.array([[-5, 0], [3, 1]]),
        'rl': ['Continuar', 'Ceder'], 'cl': ['Continuar', 'Ceder']
    },
    'Matching Pennies': {
        'A': np.array([[1, -1], [-1, 1]]),
        'B': np.array([[-1, 1], [1, -1]]),
        'rl': ['Cara', 'Cruz'], 'cl': ['Cara', 'Cruz']
    }
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, g) in zip(axes, juegos.items()):
    game = nash.Game(g['A'], g['B'])
    eqs = list(game.support_enumeration())
    types = ['puro' if max(s1) == 1 else 'mixto' for s1, s2 in eqs]
    print(f'\n{name}:')
    for (s1, s2), t in zip(eqs, types):
        print(f'  EQ {t}: J1={np.round(s1,3)}, J2={np.round(s2,3)}')
plt.tight_layout()
plt.show()

## 5. Ejercicios

In [ ]:
# EJERCICIO 1 — Encuentra todos los equilibrios de Nash
A_ej = np.array([[3, 1], [0, 2]])
B_ej = np.array([[2, 0], [1, 3]])

# TU CÓDIGO AQUÍ:
# game_ej = nash.Game(A_ej, B_ej)
# eqs_ej = list(game_ej.support_enumeration())
# print(eqs_ej)

# Test:
# assert len(eqs_ej) >= 1, 'El juego debe tener al menos un equilibrio'

In [ ]:
# EJERCICIO 2 — Visualiza las curvas de mejor respuesta del juego anterior
# y verifica que los equilibrios coinciden con las intersecciones.
# TU CÓDIGO AQUÍ:
# plot_best_responses(A_ej, B_ej, ['F1','F2'], ['C1','C2'], 'Mi juego')
# plt.show()
